In [1]:
import pathlib

# import pandas as pd
import jax.random as jr
from torch.utils.data import DataLoader
# import altair as alt

import trapallada.data
import trapallada.plot

So that imported modules are automatically reloaded

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Images

The *base* file name for the data

In [3]:
# base_filename = pathlib.Path('small.pt')
base_filename = pathlib.Path.home() / 'Sync' / 'research' / 'light curves' / 'small.pt'

`Dataset` is loaded: 18,000 gray-scale (i.e., single-channel) images

In [4]:
ds = trapallada.data.ImageDataset(
    base_filename.with_stem(base_filename.stem + '_imgs'),
    base_filename.with_stem(base_filename.stem + '_labels').with_suffix('.parquet'),
)
ds.imgs.shape

torch.Size([18000, 1, 30, 250])

There is an internal `pd.DataFrame` storing all the metadata

In [5]:
ds.df

,label,std,file name
0,Constant,1.0,Constant/cLC_s1_793.jpg
1,Constant,3.0,Constant/cLC_s3_524.jpg
2,Constant,3.0,Constant/cLC_s3_956.jpg
3,Constant,3.0,Constant/cLC_s3_283.jpg
4,Constant,5.0,Constant/cLC_s5_666.jpg
...,...,...,...
17995,AM,3.0,AM/AM_s3_613.jpg
17996,AM,1.0,AM/AM_s1_303.jpg
17997,AM,2.0,AM/AM_s2_894.jpg
17998,AM,10.0,AM/AM_s10_750.jpg


Using a `DataLoader`

In [6]:
dl = DataLoader(ds, batch_size=5)
dl

How many things are output by every batch?

In [7]:
batch = next(iter(dl))
len(batch)

3

- Indexes

- Images

- Labels

In [8]:
batch[0], batch[1].shape, batch[2]

(tensor([0, 1, 2, 3, 4]),
 torch.Size([5, 1, 30, 250]),
 tensor([1, 1, 1, 1, 1], dtype=torch.int8))

# Signals

Input directory

In [9]:

input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()

Individual files with

- the actual signals, and

- a `DataFrame` with metadata.

In [10]:
base_filename = '4_classes_modulated'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

A *PyTorch* `Dataset`

In [11]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
len(ds)

12983

It encompasses a *PyTorch* `Tensor` for the signals

In [12]:
ds.signals.shape

torch.Size([12983, 1, 10000])

and a *Pandas* `DataFrame`

In [13]:
ds.metadata_df.sample(n=10)

,label,SNR,color,file name
11351,carrier,31.620,None,singlePeriod/carrier_SNR31.62_Simul479.mat
7043,AM,10.000,None,AM/AM_SNR10_Simul528.mat
1309,FM,1.000,None,FM/FM_SNR1_Simul416.mat
7872,AM,1.000,None,AM/AM_SNR1_Simul218.mat
9131,carrier,1.000,None,singlePeriod/carrier_SNR1_Simul660.mat
10022,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul236.mat
3321,FM,31.620,None,FM/FM_SNR31.62_Simul783.mat
12918,noise,NaN,0.20551,noise/noise_color0.20551.mat
6502,AM,31.620,None,AM/AM_SNR31.62_Simul466.mat
11363,carrier,1.000,None,singlePeriod/carrier_SNR1_Simul343.mat


In [14]:
mean = ds.signals.mean(axis=-1)
std = ds.signals.std(axis=-1)
mean, std

(tensor([[-1.9975e-01],
         [-1.0158e-01],
         [ 1.4231e-04],
         ...,
         [ 2.2189e-02],
         [ 3.9760e-01],
         [ 2.2086e-02]], dtype=torch.float64),
 tensor([[0.8860],
         [0.7669],
         [0.7920],
         ...,
         [0.9930],
         [2.7855],
         [1.1968]], dtype=torch.float64))

One can also *normalize* the `DataSet`

In [15]:
normalized_ds = trapallada.data.SignalsDataset(signals_file, metadata_file,normalize=True)
mean = normalized_ds.signals.mean(axis=-1)
std = normalized_ds.signals.std(axis=-1)
mean, std

(tensor([[ 1.0658e-17],
         [ 4.9738e-18],
         [-4.9738e-18],
         ...,
         [ 5.5067e-18],
         [ 3.7303e-17],
         [ 1.3500e-17]], dtype=torch.float64),
 tensor([[1.0000],
         [1.0000],
         [1.0000],
         ...,
         [1.0000],
         [1.0000],
         [1.0000]], dtype=torch.float64))

Using a `DataLoader`

In [16]:
dl = DataLoader(ds, batch_size=5)
dl

How many things are output by every batch?

In [17]:
batch = next(iter(dl))
len(batch)

3

- Indexes

- Images

- Labels

In [18]:
batch[0], batch[1].shape, batch[2]

(tensor([0, 1, 2, 3, 4]),
 torch.Size([5, 1, 10000]),
 tensor([1, 1, 1, 1, 1], dtype=torch.int8))

Let us plot the first

In [19]:
n_samples_to_plot = 1_000

samples of the first signal in the batch

In [20]:
trapallada.plot.signal(batch[1][0][0, 0:n_samples_to_plot].numpy())

alt.Chart(...)

Some random signals

In [21]:
n_signals = 15
i_signals = jr.randint(jr.PRNGKey(42), (n_signals,), 0, len(ds)).tolist()
signals = ds.signals[i_signals]
signals_df = ds.metadata_df.iloc[i_signals]
signals_df

,label,SNR,color,file name
12427,noise,NaN,1.8077,noise/noise_color1.8077.mat
7770,AM,3.162,None,AM/AM_SNR3.162_Simul819.mat
1120,FM,1.000,None,FM/FM_SNR1_Simul538.mat
12508,noise,NaN,1.122,noise/noise_color1.122.mat
12227,noise,NaN,0.6046,noise/noise_color0.6046.mat
11677,carrier,31.620,None,singlePeriod/carrier_SNR31.62_Simul798.mat
9306,carrier,1.000,None,singlePeriod/carrier_SNR1_Simul518.mat
11353,carrier,3.162,None,singlePeriod/carrier_SNR3.162_Simul822.mat
5483,AM,10.000,None,AM/AM_SNR10_Simul659.mat
285,FM,1.000,None,FM/FM_SNR1_Simul138.mat


In [22]:
trapallada.plot.signals_dataset(ds, n_samples=1_000)

alt.Chart(...)